# 🐋 DeepWhale — Exploratory Data Analysis

**Behavioral Analysis and Classification of Crypto Whales**

This notebook provides a complete EDA of on-chain ETH whale transactions,
covering:

1. Raw transaction overview
2. Transaction size distribution
3. Temporal patterns (time-of-day, day-of-week)
4. Top whale addresses & exchange interactions
5. Network topology (transaction graph)
6. Address-level feature profiles
7. Whale type label distribution
8. Correlation heatmap
9. ETH price vs whale activity
10. Anomaly score distribution

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import networkx as nx
from pathlib import Path

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linewidth': 0.5,
})
ACCENT = '#3498db'
ACCENT2 = '#e74c3c'
ACCENT3 = '#2ecc71'

DATA_DIR = Path('data')
print('Setup complete.')

---
## 1. Raw Transaction Data Overview

In [ ]:
raw_path = DATA_DIR / 'raw_whale_transactions.csv'

if not raw_path.exists():
    print('raw_whale_transactions.csv not found.\nRun: python data_collection.py')
else:
    df = pd.read_csv(raw_path, parse_dates=['timestamp'])
    print(f'Rows: {len(df):,}')
    print(f'Unique from-addresses (whale senders): {df["from_address"].nunique():,}')
    print(f'Unique to-addresses (receivers):        {df["to_address"].nunique():,}')
    print(f'Date range: {df["timestamp"].min()} → {df["timestamp"].max()}')
    print(f'Total ETH moved:  {df["value_eth"].sum():,.2f} ETH')
    if df['value_usd'].notna().any():
        print(f'Total USD moved:  ${df["value_usd"].sum():,.0f}')
    display(df.head())

In [ ]:
display(df[['value_eth', 'value_usd', 'gas_price_gwei', 'eth_price_usd']].describe().round(2))

---
## 2. Transaction Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram
ax = axes[0]
sns.histplot(df['value_eth'], bins=50, log_scale=True, color=ACCENT, ax=ax)
ax.set_xlabel('ETH Transferred (log scale)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Whale Transaction Sizes')
ax.axvline(df['value_eth'].median(), color=ACCENT3, linestyle='--', label=f'Median: {df["value_eth"].median():.1f} ETH')
ax.axvline(df['value_eth'].mean(), color=ACCENT2, linestyle='--', label=f'Mean: {df["value_eth"].mean():.1f} ETH')
ax.legend()

# Cumulative distribution
ax = axes[1]
sorted_eth = np.sort(df['value_eth'])
cumsum = np.cumsum(sorted_eth)
ax.plot(sorted_eth, cumsum / cumsum[-1], color=ACCENT, linewidth=2)
ax.fill_between(sorted_eth, cumsum / cumsum[-1], alpha=0.2, color=ACCENT)
ax.set_xscale('log')
ax.set_xlabel('ETH Transferred (log scale)')
ax.set_ylabel('Cumulative fraction of total volume')
ax.set_title('Cumulative ETH Volume Distribution')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_size_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Concentration: top 10% of txs account for X% of volume
p90 = np.percentile(df['value_eth'], 90)
top10_vol = df[df['value_eth'] >= p90]['value_eth'].sum()
print(f'Top 10% of transactions (≥ {p90:.0f} ETH) account for {top10_vol/df["value_eth"].sum()*100:.1f}% of total volume')

---
## 3. Temporal Patterns

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['date'] = df['timestamp'].dt.date

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hour-of-day
ax = axes[0]
hour_counts = df.groupby('hour')['value_eth'].sum()
ax.bar(hour_counts.index, hour_counts.values, color=ACCENT, alpha=0.8)
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Total ETH')
ax.set_title('Whale Activity by Hour of Day')
ax.set_xticks(range(0, 24, 3))

# Day of week
ax = axes[1]
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = df.groupby('day_of_week')['value_eth'].sum().reindex(dow_order, fill_value=0)
bars = ax.bar(range(7), dow_counts.values, color=ACCENT3, alpha=0.8)
ax.set_xticks(range(7))
ax.set_xticklabels([d[:3] for d in dow_order], rotation=30)
ax.set_ylabel('Total ETH')
ax.set_title('Whale Activity by Day of Week')

# Daily whale tx count over time
ax = axes[2]
daily = df.groupby('date').size().reset_index(name='count')
daily['date'] = pd.to_datetime(daily['date'])
ax.plot(daily['date'], daily['count'], color=ACCENT2, linewidth=1.5)
ax.fill_between(daily['date'], daily['count'], alpha=0.2, color=ACCENT2)
ax.set_xlabel('Date')
ax.set_ylabel('# Whale Transactions')
ax.set_title('Daily Whale Transaction Count')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_temporal.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 4. Top Whale Addresses & Exchange Interactions

In [ ]:
known_path = DATA_DIR / 'known_addresses.csv'
if known_path.exists():
    known_df = pd.read_csv(known_path)
    known_set = set(known_df['address'].str.lower())
    known_labels = dict(zip(known_df['address'].str.lower(), known_df['label']))
else:
    known_set = set()
    known_labels = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 senders by volume
ax = axes[0]
top_senders = df.groupby('from_address')['value_eth'].sum().sort_values(ascending=False).head(15)
colors = [ACCENT2 if a.lower() in known_set else ACCENT for a in top_senders.index]
short_labels = [f'{a[:6]}...{a[-4:]}' for a in top_senders.index]
ax.barh(short_labels[::-1], top_senders.values[::-1], color=colors[::-1], alpha=0.85)
ax.set_xlabel('Total ETH Sent')
ax.set_title('Top 15 Whale Senders (red = known exchange)')

# Top 15 receivers
ax = axes[1]
top_receivers = df.groupby('to_address')['value_eth'].sum().sort_values(ascending=False).head(15)
colors_r = [ACCENT2 if a.lower() in known_set else ACCENT3 for a in top_receivers.index]
recv_labels = [known_labels.get(a.lower(), f'{a[:6]}...{a[-4:]}') for a in top_receivers.index]
ax.barh(recv_labels[::-1], top_receivers.values[::-1], color=colors_r[::-1], alpha=0.85)
ax.set_xlabel('Total ETH Received')
ax.set_title('Top 15 Receivers (red = known exchange)')

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_top_addresses.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Exchange destination ratio
df['to_exchange'] = df['to_address'].str.lower().isin(known_set)
exch_pct = df['to_exchange'].mean() * 100
exch_vol_pct = df.loc[df['to_exchange'], 'value_eth'].sum() / df['value_eth'].sum() * 100
print(f'{exch_pct:.1f}% of transactions go to known exchange addresses')
print(f'{exch_vol_pct:.1f}% of total whale volume flows to known exchanges')

---
## 5. Transaction Network Topology

In [ ]:
# Build directed graph — limit to top 80 nodes for readability
top_addrs = set(
    df['from_address'].value_counts().head(40).index.tolist() +
    df['to_address'].value_counts().head(40).index.tolist()
)
df_net = df[df['from_address'].isin(top_addrs) & df['to_address'].isin(top_addrs)]

G = nx.from_pandas_edgelist(
    df_net, source='from_address', target='to_address',
    edge_attr=['value_eth'], create_using=nx.DiGraph()
)

plt.figure(figsize=(13, 13), facecolor='#0d1117')
ax = plt.gca()
ax.set_facecolor('#0d1117')

degrees = dict(G.degree)
node_sizes = [max(30, degrees.get(n, 1) * 60) for n in G.nodes()]
node_colors = ['#e74c3c' if n.lower() in known_set else '#3498db' for n in G.nodes()]

pos = nx.spring_layout(G, k=0.25, iterations=50, seed=42)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.85)
nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color='#8b949e', arrows=True,
                       arrowsize=10, connectionstyle='arc3,rad=0.1')

# Label only exchange nodes
exchange_nodes = {n: known_labels.get(n.lower(), '')[:12] for n in G.nodes() if n.lower() in known_set}
nx.draw_networkx_labels(G, pos, labels=exchange_nodes, font_size=7, font_color='white')

plt.title('Whale Transaction Network (red = known exchange, size = degree)', fontsize=14, color='white')
plt.axis('off')
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_network.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Density: {nx.density(G):.4f}')

---
## 6. Address Feature Profiles

In [ ]:
feat_path = DATA_DIR / 'address_features.csv'
labeled_path = DATA_DIR / 'labeled_addresses.csv'

src = labeled_path if labeled_path.exists() else feat_path
if not src.exists():
    print(f'{src} not found. Run feature_engineering.py and labeling.py first.')
else:
    feat_df = pd.read_csv(src)
    print(f'Feature matrix: {feat_df.shape}')
    display(feat_df.describe().round(3))

In [ ]:
if 'feat_df' in dir():
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    scatter_pairs = [
        ('total_eth_out', 'unique_receivers'),
        ('tx_count_out', 'exchange_ratio'),
        ('avg_tx_eth', 'gas_variability'),
        ('active_days', 'net_flow_eth'),
        ('hour_entropy', 'round_number_ratio'),
        ('tx_count_out', 'top1_receiver_ratio'),
    ]

    label_col = 'label' if 'label' in feat_df.columns else None
    palette = {'exchange_depositor': '#3498db', 'accumulator': '#2ecc71',
               'active_trader': '#e74c3c', 'unknown_whale': '#95a5a6'}

    for ax, (xcol, ycol) in zip(axes, scatter_pairs):
        if xcol not in feat_df.columns or ycol not in feat_df.columns:
            ax.set_visible(False)
            continue
        if label_col:
            for lbl, grp in feat_df.groupby(label_col):
                ax.scatter(grp[xcol], grp[ycol], label=lbl, alpha=0.7, s=40,
                           color=palette.get(lbl, '#888'), edgecolors='white', linewidths=0.3)
            ax.legend(fontsize=7, framealpha=0.3)
        else:
            ax.scatter(feat_df[xcol], feat_df[ycol], alpha=0.7, s=40, color=ACCENT)
        ax.set_xlabel(xcol)
        ax.set_ylabel(ycol)
        ax.set_title(f'{xcol} vs {ycol}')

    plt.suptitle('Whale Feature Scatter Plots', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'eda_feature_scatter.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

---
## 7. Whale Type Label Distribution

In [ ]:
if 'feat_df' in dir() and 'label' in feat_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Count distribution
    ax = axes[0]
    counts = feat_df['label'].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=[palette.get(l, '#888') for l in counts.index], alpha=0.85)
    ax.set_ylabel('Number of Addresses')
    ax.set_title('Whale Type Distribution (count)')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', va='bottom', color='white', fontsize=10)
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

    # Volume by label
    ax = axes[1]
    if 'total_eth_out' in feat_df.columns:
        vol = feat_df.groupby('label')['total_eth_out'].sum().sort_values(ascending=True)
        ax.barh(vol.index, vol.values, color=[palette.get(l, '#888') for l in vol.index], alpha=0.85)
        ax.set_xlabel('Total ETH Out')
        ax.set_title('ETH Volume by Whale Type')

    plt.tight_layout()
    plt.savefig(DATA_DIR / 'eda_labels.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

---
## 8. Feature Correlation Heatmap

In [ ]:
if 'feat_df' in dir():
    numeric_cols = feat_df.select_dtypes(include='number').columns.tolist()
    drop_cols = ['is_known_exchange', 'label_confidence', 'block_span']
    numeric_cols = [c for c in numeric_cols if c not in drop_cols]

    corr = feat_df[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(13, 11))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
        center=0, linewidths=0.4, linecolor='#21262d',
        cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 8}
    )
    ax.set_title('Feature Correlation Heatmap', fontsize=14)
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'eda_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

---
## 9. ETH Price vs Whale Activity

In [ ]:
if 'df' in dir() and df['eth_price_usd'].notna().sum() > 5:
    df_price = df.dropna(subset=['eth_price_usd']).copy()
    df_price['hour'] = df_price['timestamp'].dt.floor('h')

    hourly = df_price.groupby('hour').agg(
        tx_count=('tx_hash', 'count'),
        total_eth=('value_eth', 'sum'),
        avg_price=('eth_price_usd', 'mean')
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.set_facecolor('#0d1117')

    ax1.bar(hourly['hour'], hourly['total_eth'], color=ACCENT, alpha=0.6, label='Whale ETH moved')
    ax1.set_xlabel('Time (UTC)')
    ax1.set_ylabel('ETH Volume', color=ACCENT)
    ax1.tick_params(axis='y', colors=ACCENT)

    ax2 = ax1.twinx()
    ax2.plot(hourly['hour'], hourly['avg_price'], color='#f1c40f', linewidth=2, label='ETH/USD Price')
    ax2.set_ylabel('ETH Price (USD)', color='#f1c40f')
    ax2.tick_params(axis='y', colors='#f1c40f')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', framealpha=0.3)

    ax1.set_title('Hourly Whale ETH Volume vs ETH/USD Price', fontsize=13)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'eda_price_vs_activity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

    # Correlation
    hourly['price_change'] = hourly['avg_price'].pct_change()
    corr_val = hourly['total_eth'].corr(hourly['price_change'])
    print(f'Pearson correlation (whale ETH volume vs ETH price change): {corr_val:.3f}')
else:
    print('ETH price data not available in raw transactions (Binance API may have failed during collection).')

---
## 10. Anomaly Score Distribution

In [ ]:
anomaly_path = DATA_DIR / 'anomaly_scores.csv'
if not anomaly_path.exists():
    print('anomaly_scores.csv not found. Run anomaly_detection.py first.')
else:
    anom_df = pd.read_csv(anomaly_path)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Score distribution
    ax = axes[0]
    normal = anom_df.loc[anom_df['anomaly_label'] == 1, 'anomaly_score']
    anomal = anom_df.loc[anom_df['anomaly_label'] == -1, 'anomaly_score']
    bins = np.linspace(anom_df['anomaly_score'].min(), anom_df['anomaly_score'].max(), 35)
    ax.hist(normal, bins=bins, color=ACCENT, alpha=0.7, label='Normal')
    ax.hist(anomal, bins=bins, color=ACCENT2, alpha=0.8, label='Anomalous')
    ax.set_xlabel('Isolation Forest Score')
    ax.set_ylabel('Count')
    ax.set_title('Anomaly Score Distribution')
    ax.legend(framealpha=0.3)

    # ETH sent vs anomaly score
    ax = axes[1]
    if 'total_eth_out' in anom_df.columns:
        colors_a = [ACCENT2 if l == -1 else ACCENT for l in anom_df['anomaly_label']]
        ax.scatter(anom_df['anomaly_score'], np.log1p(anom_df['total_eth_out']),
                   c=colors_a, alpha=0.7, s=40, edgecolors='white', linewidths=0.3)
        ax.set_xlabel('Anomaly Score')
        ax.set_ylabel('log(1 + Total ETH Out)')
        ax.set_title('Anomaly Score vs ETH Volume')
        ax.axvline(anomal.max(), color='orange', linestyle='--', lw=1.2, label='Threshold')
        ax.legend(framealpha=0.3)

    plt.tight_layout()
    plt.savefig(DATA_DIR / 'eda_anomaly.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()

    n_anom = (anom_df['anomaly_label'] == -1).sum()
    print(f'Anomalous addresses: {n_anom} / {len(anom_df)} ({n_anom/len(anom_df)*100:.1f}%)')
    print('\nTop 5 most anomalous:')
    display(anom_df[anom_df['anomaly_label'] == -1]
            .sort_values('anomaly_score')[['address','anomaly_score','tx_count_out','total_eth_out']]
            .head(5))

---
## Summary

| Module | Status |
|--------|--------|
| Data collection (`data_collection.py`) | Collects 500+ blocks, saves CSV |
| Feature engineering (`feature_engineering.py`) | 22 per-address behavioural features |
| Labeling (`labeling.py`) | 4-class heuristic labels |
| Clustering (`clustering.py`) | KMeans + DBSCAN + PCA/t-SNE |
| Classification (`classification.py`) | XGBoost + RF + SHAP |
| Anomaly detection (`anomaly_detection.py`) | Isolation Forest + price correlation |
| Dashboard (`dashboard.py`) | Streamlit — `streamlit run dashboard.py` |

### How to run the full pipeline

```bash
# 1. Collect on-chain data (needs ALCHEMY_URL in .env)
python data_collection.py

# 2. Build address-level features
python feature_engineering.py

# 3. Assign heuristic labels
python labeling.py

# 4. Cluster whale archetypes
python clustering.py

# 5. Train classifier
python classification.py

# 6. Detect anomalies
python anomaly_detection.py

# 7. Launch interactive dashboard
streamlit run dashboard.py
```